In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging
import sys

# Clear any existing handlers to prevent duplicates
root_logger = logging.getLogger()
if root_logger.handlers:
    for handler in root_logger.handlers:
        root_logger.removeHandler(handler)

# Configure logging FIRST before importing DataLoader
logging.basicConfig(level=logging.WARNING, force=True)

# Suppress specific loggers that are verbose
for logger_name in ['__main__', 'snowflake', 'azure', 'urllib3']:
    logging.getLogger(logger_name).setLevel(logging.WARNING)

# Add the project root to the path to import custom modules
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import the DataLoader class
from scripts.data_preparation.load import DataLoader

print("✓ Libraries imported successfully")
print("✓ Logging configured (INFO messages suppressed)")

✓ Libraries imported successfully
✓ Logging configured (INFO messages suppressed)


In [7]:
# Getting email data from startdate to enddate
startdate = '2026-05-01'
enddate = '2026-05-31' # '2026-05-11'

print(f"Tracking Email History will be Retrieved from {startdate} to {enddate}")

Tracking Email History will be Retrieved from 2026-05-01 to 2026-05-31


In [8]:
# Force reload the DataLoader module to ensure latest code is used
import importlib
import sys

# Remove from cache if present, then reimport fresh
if 'scripts.data_preparation.load' in sys.modules:
    del sys.modules['scripts.data_preparation.load']

from scripts.data_preparation.load import DataLoader
print("✓ DataLoader module loaded fresh")

# Verify the parameter substitution method being used
import inspect
src = inspect.getsource(DataLoader.load_from_snowflake)
if 'query.replace' in src:
    print("✓ Using str.replace() for parameter substitution (correct)")
elif 'query.format' in src:
    print("⚠️  Still using str.format() - reload did not pick up changes!")


✓ DataLoader module loaded fresh
⚠️  Still using str.format() - reload did not pick up changes!


## 1. Load Tracking CheckCall Data from Snowflake

Load the tracking checkcall data from Snowflake using the DataLoader class.

In [ ]:
# Execute the query to get historical load data
print("Querying Snowflake for Tracking checkcall data...")
print(f"Date Range: {startdate} to {enddate}")

loader = DataLoader()

df = loader.load_from_snowflake(
    sql_file_path='../sql/loadTrackingOverprocessing2.sql',
    params={'startdate': startdate, 'enddate': enddate}
)

# Convert datetime column immediately after load (handles mixed types / NaN floats)
df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], utc=True, errors='coerce')

print(f"\n✓ Loaded {len(df):,} rows from Snowflake")
print(f"  Date range in data: {df['ENTERED_DATETIME_CST'].min()} to {df['ENTERED_DATETIME_CST'].max()}")
print(f"  Columns: {len(df.columns)}")
df.head()


Querying Snowflake for Tracking checkcall data...
Date Range: 2026-05-01 to 2026-05-31


/home/azureuser/loadTrackingOverProcessing/LoadTrackingOverProcessing/scripts/data_preparation/load.py:169: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, self.snowflake_conn)


In [ ]:
df[df['LOAD_NUM']==552545690] # 554287901

,LOAD_NUM,CHECK_CALL_TYPE,DESCRIPTION,ENTERED_DATETIME_CST,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE,AUTOMATED,...,TRACKING_METHOD_TYPE,APPT_TYPE,OG_SCHEDDATETIME,OG_APPTOPEN,IS_CROSS_BORDER_RELATED,TRACKING_SLA_PRE_PICK,TRACKING_SLA_IN_TRANSIT,TRACKING_SLA_TOTAL,TRACKING_SLA_TOTAL_SCORE_ACTUAL,TRACKING_SLA_TOTAL_SCORE_POSSIBLE
14010188,552545690,CC,Check Call,2026-05-11 16:01:00+00:00,DALLAS,TX,US,32.685210,-96.885550,True,...,GPS,None,2026-05-05 09:27:00,2026-05-09 19:30:00-05:00,0,1.0,1.0,1.0,9.0,9.0
14026746,552545690,CC,Check Call,2026-05-11 16:16:00+00:00,DUNCANVILLE,TX,US,32.676410,-96.891520,True,...,GPS,None,2026-05-05 09:27:00,2026-05-09 19:30:00-05:00,0,1.0,1.0,1.0,9.0,9.0
14044331,552545690,CC,Check Call,2026-05-11 16:31:00+00:00,HUTCHINS,TX,US,32.649770,-96.708110,True,...,GPS,None,2026-05-05 09:27:00,2026-05-09 19:30:00-05:00,0,1.0,1.0,1.0,9.0,9.0
14061661,552545690,CC,Check Call,2026-05-11 16:46:00+00:00,WILMER,TX,US,32.584610,-96.678240,True,...,GPS,None,2026-05-05 09:27:00,2026-05-09 19:30:00-05:00,0,1.0,1.0,1.0,9.0,9.0
14087835,552545690,CC,Check Call,2026-05-11 17:06:00+00:00,WILMER,TX,US,32.580040,-96.675620,True,...,GPS,None,2026-05-05 09:27:00,2026-05-09 19:30:00-05:00,0,1.0,1.0,1.0,9.0,9.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15609958,552545690,CC,Check Call,2026-05-12 17:50:00+00:00,MERCEDES,TX,US,26.159460,-97.865970,True,...,GPS,None,NaT,None,0,1.0,1.0,1.0,9.0,9.0
15611586,552545690,DD,Depart Drop,2026-05-12 17:52:00+00:00,WESLACO,TX,US,26.165899,-97.998901,True,...,GPS,None,NaT,None,0,1.0,1.0,1.0,9.0,9.0
15627811,552545690,CC,Check Call,2026-05-12 18:02:00+00:00,SAN BENITO,TX,US,26.150290,-97.668120,True,...,GPS,None,NaT,None,0,1.0,1.0,1.0,9.0,9.0
15739696,552545690,D-Open,W4929838,2026-05-12 19:30:00+00:00,WESLACO,TX,United States,26.165899,-97.998901,None,...,None,RESCHEDULES SET,NaT,None,0,1.0,1.0,1.0,9.0,9.0


In [ ]:
def get_loads_with_manual_cc_before_pickup(df, early_threshold_hours=4):
    """
    Identifies LOAD_NUMs that have at least one manual check call
    (CHECK_CALL_TYPE = 'CC' and AUTOMATED = False) recorded before
    the original pickup open appointment (OG_APPTOPEN).

    Uses OG_APPTOPEN rather than the ENTERED_DATETIME_CST of the first P-Open
    event, because the appointment time in effect at the time of the check call
    may differ from when the P-Open was eventually recorded (e.g. after reschedules).
    OG_APPTOPEN is the appointment open time that was active when the CC was entered.
    Note: a single load may have multiple different OG_APPTOPEN values across its
    manual CCs as appointments are rescheduled, so it is not included in the output.

    Excludes check calls where UPDATE_USER = 'DATASCIENCE'.
    Only includes check calls where HUMAN_ENTERED_CHECKCALL_FLAG = 1.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake containing columns:
                           LOAD_NUM, CHECK_CALL_TYPE, AUTOMATED, ENTERED_DATETIME_CST,
                           UPDATE_USER, HUMAN_ENTERED_CHECKCALL_FLAG, OG_APPTOPEN
        early_threshold_hours (int): Hours before OG_APPTOPEN to classify as "very early". Default 4.

    Returns:
        pd.DataFrame: DataFrame of qualifying LOAD_NUMs with supporting details:
                      - LOAD_NUM
                      - manual_cc_before_pickup_count: number of manual CCs before OG_APPTOPEN
                      - earliest_manual_cc: datetime of the first manual CC before OG_APPTOPEN
                      - manual_cc_early_count: manual CCs more than `early_threshold_hours`
                                               before OG_APPTOPEN (very premature touches)
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], utc=True, errors='coerce')
    df['OG_APPTOPEN'] = pd.to_datetime(df['OG_APPTOPEN'], utc=True, errors='coerce')

    # Get manual check calls (CC + not automated + not DATASCIENCE user + human entered)
    # Include OG_APPTOPEN — the appointment open time active at the time of each CC
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1) &
        (df['OG_APPTOPEN'].notna())
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST', 'OG_APPTOPEN']].copy()

    # Keep only manual CCs that occurred BEFORE their row's OG_APPTOPEN
    manual_cc_before = manual_cc[
        manual_cc['ENTERED_DATETIME_CST'] < manual_cc['OG_APPTOPEN']
    ].copy()

    # Flag CCs that are more than `early_threshold_hours` before OG_APPTOPEN
    early_cutoff = pd.Timedelta(hours=early_threshold_hours)
    manual_cc_before['is_very_early'] = (
        manual_cc_before['OG_APPTOPEN'] - manual_cc_before['ENTERED_DATETIME_CST']
    ) > early_cutoff

    # Aggregate per load
    result = (
        manual_cc_before.groupby('LOAD_NUM')
        .agg(
            manual_cc_before_pickup_count=('ENTERED_DATETIME_CST', 'count'),
            earliest_manual_cc=('ENTERED_DATETIME_CST', 'min'),
            manual_cc_early_count=('is_very_early', 'sum')
        )
        .reset_index()
    )

    result['manual_cc_early_count'] = result['manual_cc_early_count'].astype(int)
    result = result.sort_values('manual_cc_before_pickup_count', ascending=False)

    return result


# Run the function
total_loads = df['LOAD_NUM'].nunique()
loads_with_early_manual_cc = get_loads_with_manual_cc_before_pickup(df, early_threshold_hours=4)
qualifying_loads = len(loads_with_early_manual_cc)
total_before_pickup = loads_with_early_manual_cc['manual_cc_before_pickup_count'].sum()
total_very_early = loads_with_early_manual_cc['manual_cc_early_count'].sum()

print(f"Total distinct LOAD_NUMs in dataset:               {total_loads:,}")
print(f"LOAD_NUMs with manual CC before OG_APPTOPEN:       {qualifying_loads:,}")
print(f"Percentage:                                        {qualifying_loads / total_loads * 100:.1f}%")
print(f"Total manual CCs before OG_APPTOPEN:               {total_before_pickup:,}")
print(f"  of which >4 hrs before OG_APPTOPEN:              {total_very_early:,} ({total_very_early / total_before_pickup * 100:.1f}%)")
loads_with_early_manual_cc.head(10)


Total distinct LOAD_NUMs in dataset:               311,086
LOAD_NUMs with manual CC before pickup open:       14,860
Percentage:                                        4.8%
Total manual CCs before pickup open:               16,337
  of which >4 hrs before pickup open:              7,721 (47.3%)


,LOAD_NUM,first_pickup_open,manual_cc_before_pickup_count,earliest_manual_cc,manual_cc_early_count
14716,555191784,2026-05-29 19:00:00+00:00,8,2026-05-28 22:43:00+00:00,8
7989,553044827,2026-05-15 16:00:00+00:00,7,2026-05-11 20:42:00+00:00,7
52,548544546,2026-05-11 14:00:00+00:00,6,2026-05-05 22:48:00+00:00,6
13027,554287901,2026-05-22 15:00:00+00:00,6,2026-05-22 00:54:00+00:00,4
2150,551828882,2026-05-07 20:00:00+00:00,5,2026-05-06 22:24:00+00:00,2
8600,553198002,2026-05-12 18:00:00+00:00,4,2026-05-12 13:33:00+00:00,1
11897,553959910,2026-05-29 18:00:00+00:00,4,2026-05-27 12:56:00+00:00,3
6678,552817819,2026-05-15 00:00:00+00:00,4,2026-05-14 14:45:00+00:00,1
2709,551986260,2026-05-15 00:00:00+00:00,4,2026-05-14 12:29:00+00:00,2
11212,553765055,2026-05-22 16:30:00+00:00,4,2026-05-22 13:34:00+00:00,0


In [ ]:
print("Total Manual CCs before Pickup Open:", loads_with_early_manual_cc['manual_cc_before_pickup_count'].sum())

Total Manual CCs before Pickup Open: 16337


In [7]:
def get_manual_cc_near_auto_in_transit(df, window_minutes=30):
    """
    Identifies per load how many manual check calls occurred during active transit
    (between first P-Open and last D-Close) AND were sandwiched between automated
    check calls — i.e., there is an auto CC within `window_minutes` BEFORE the
    manual CC AND an auto CC within `window_minutes` AFTER it.

    The intent is to measure over-processing: manual CCs entered by a dispatcher
    when automated tracking was already firing on schedule around the same time.

    Criteria for a manual check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1

    Criteria for an automated check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == True
        - HUMAN_ENTERED_CHECKCALL_FLAG == 0
        - UPDATE_USER == 'DATASCIENCE'

    Criteria for "sandwiched by auto CCs":
        - At least one auto CC on the same load within `window_minutes` BEFORE the manual CC
        - At least one auto CC on the same load within `window_minutes` AFTER the manual CC

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.
        window_minutes (int): Proximity window in minutes. Default 60.

    Returns:
        pd.DataFrame: One row per load with:
            - LOAD_NUM
            - first_pickup_open: earliest P-Open datetime
            - last_dropoff_close: latest D-Close datetime
            - transit_manual_cc_total: total manual CCs in transit window
            - manual_cc_sandwiched_count: manual CCs with an auto CC before AND after within window
            - pct_sandwiched: percentage of transit manual CCs that were sandwiched
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], utc=True, errors='coerce')
    window = pd.Timedelta(minutes=window_minutes)

    # --- Transit window boundaries per load ---
    first_pickup = (
        df[df['CHECK_CALL_TYPE'] == 'P-Open']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .min()
        .rename('first_pickup_open')
    )

    last_dropoff = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
    )

    transit_bounds = pd.concat([first_pickup, last_dropoff], axis=1).dropna().reset_index()

    # --- Manual CCs ---
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].copy()

    # Keep only manual CCs inside the transit window
    manual_cc = manual_cc.merge(transit_bounds, on='LOAD_NUM', how='inner')
    manual_cc = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] >= manual_cc['first_pickup_open']) &
        (manual_cc['ENTERED_DATETIME_CST'] <= manual_cc['last_dropoff_close'])
    ].copy()

    # --- Automated CCs: CC type, automated flag, no human entry, UPDATE_USER == 'DATASCIENCE' ---
    auto_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == True) &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 0) &
        (df['UPDATE_USER'] != 'DATASCIENCE')
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].rename(
        columns={'ENTERED_DATETIME_CST': 'auto_datetime'}
    ).copy()

    # --- Cross-join per load to check sandwich condition ---
    # signed_diff > 0: auto CC is AFTER the manual CC
    # signed_diff < 0: auto CC is BEFORE the manual CC
    merged = manual_cc.merge(auto_cc, on='LOAD_NUM', how='left')
    merged['signed_diff'] = merged['auto_datetime'] - merged['ENTERED_DATETIME_CST']

    merged['auto_before'] = (merged['signed_diff'] < pd.Timedelta(0)) & \
                            (merged['signed_diff'].abs() <= window)
    merged['auto_after']  = (merged['signed_diff'] > pd.Timedelta(0)) & \
                            (merged['signed_diff'].abs() <= window)

    # Per manual CC: require at least one auto before AND one auto after
    sandwich_flags = (
        merged.groupby(['LOAD_NUM', 'ENTERED_DATETIME_CST'])
        .agg(
            has_auto_before=('auto_before', 'any'),
            has_auto_after=('auto_after', 'any')
        )
        .reset_index()
    )
    sandwich_flags['sandwiched'] = sandwich_flags['has_auto_before'] & sandwich_flags['has_auto_after']

    manual_cc = manual_cc.merge(
        sandwich_flags[['LOAD_NUM', 'ENTERED_DATETIME_CST', 'sandwiched']],
        on=['LOAD_NUM', 'ENTERED_DATETIME_CST'],
        how='left'
    )
    manual_cc['sandwiched'] = manual_cc['sandwiched'].fillna(False)

    # --- Aggregate per load ---
    result = (
        manual_cc.groupby(['LOAD_NUM', 'first_pickup_open', 'last_dropoff_close'])
        .agg(
            transit_manual_cc_total=('ENTERED_DATETIME_CST', 'count'),
            manual_cc_sandwiched_count=('sandwiched', 'sum')
        )
        .reset_index()
    )

    result['manual_cc_sandwiched_count'] = result['manual_cc_sandwiched_count'].astype(int)
    result['pct_sandwiched'] = (
        result['manual_cc_sandwiched_count'] / result['transit_manual_cc_total'] * 100
    ).round(1)

    result = result.sort_values('manual_cc_sandwiched_count', ascending=False)

    return result


# Run the function
in_transit_overprocessing = get_manual_cc_near_auto_in_transit(df, window_minutes=30)
qualifying_loads = len(in_transit_overprocessing)
total_sandwiched = in_transit_overprocessing['manual_cc_sandwiched_count'].sum()
total_transit_manual = in_transit_overprocessing['transit_manual_cc_total'].sum()

print(f"Loads with manual CCs in transit window:           {qualifying_loads:,}")
print(f"Total manual CCs in transit window:                {total_transit_manual:,}")
print(f"Manual CCs sandwiched by auto CCs (±30 min):      {total_sandwiched:,}")
print(f"Overall % sandwiched:                              {total_sandwiched / total_transit_manual * 100:.1f}%")
in_transit_overprocessing.head(10)


Loads with manual CCs in transit window:           14,239
Total manual CCs in transit window:                18,965
Manual CCs sandwiched by auto CCs (±30 min):      8,495
Overall % sandwiched:                              44.8%


,LOAD_NUM,first_pickup_open,last_dropoff_close,transit_manual_cc_total,manual_cc_sandwiched_count,pct_sandwiched
801,551412317,2026-05-21 15:00:00+00:00,2026-05-27 16:00:00+00:00,19,19,100.0
10839,553916777,2026-05-21 15:00:00+00:00,2026-05-26 14:00:00+00:00,18,17,94.4
8908,553442682,2026-05-26 15:00:00+00:00,2026-06-01 12:30:00+00:00,20,17,85.0
483,551098226,2026-05-01 15:00:00+00:00,2026-05-04 16:00:00+00:00,16,16,100.0
1317,551733783,2026-05-22 15:00:00+00:00,2026-05-26 18:00:00+00:00,15,15,100.0
5819,552771412,2026-05-08 15:00:00+00:00,2026-05-12 22:00:00+00:00,15,15,100.0
1321,551733805,2026-05-01 15:00:00+00:00,2026-05-05 04:00:00+00:00,16,15,93.8
12644,554454971,2026-05-26 14:00:00+00:00,2026-05-28 16:00:00+00:00,14,14,100.0
21,546748574,2026-05-15 15:00:00+00:00,2026-05-19 04:00:00+00:00,14,14,100.0
4052,552423490,2026-05-08 20:00:00+00:00,2026-05-12 03:00:00+00:00,14,14,100.0


In [8]:
def get_loads_with_manual_cc_after_dropoff(df):
    """
    Identifies LOAD_NUMs that have at least one manual check call
    (CHECK_CALL_TYPE = 'CC' and AUTOMATED = False) recorded AFTER
    the last drop off close event (CHECK_CALL_TYPE = 'D-Close').
    Excludes check calls where UPDATE_USER = 'DATASCIENCE'.
    Only includes check calls where HUMAN_ENTERED_CHECKCALL_FLAG = 1.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake containing columns:
                           LOAD_NUM, CHECK_CALL_TYPE, AUTOMATED, ENTERED_DATETIME_CST,
                           UPDATE_USER, HUMAN_ENTERED_CHECKCALL_FLAG

    Returns:
        pd.DataFrame: DataFrame of qualifying LOAD_NUMs with supporting details:
                      - LOAD_NUM
                      - last_dropoff_close: latest D-Close datetime
                      - manual_cc_after_dropoff_count: number of manual CCs after D-Close
                      - latest_manual_cc: datetime of the last manual CC after D-Close
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], utc=True, errors='coerce')

    # Get the last D-Close datetime per load
    dropoff_close = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
        .reset_index()
    )

    # Get manual check calls (CC + not automated + not DATASCIENCE user + human entered)
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].copy()

    # Join manual CCs with their load's last D-Close datetime
    manual_cc = manual_cc.merge(dropoff_close, on='LOAD_NUM', how='inner')

    # Keep only manual CCs that occurred AFTER the last D-Close
    manual_cc_after = manual_cc[
        manual_cc['ENTERED_DATETIME_CST'] > manual_cc['last_dropoff_close']
    ]

    # Aggregate per load
    result = (
        manual_cc_after.groupby(['LOAD_NUM', 'last_dropoff_close'])
        .agg(
            manual_cc_after_dropoff_count=('ENTERED_DATETIME_CST', 'count'),
            latest_manual_cc=('ENTERED_DATETIME_CST', 'max')
        )
        .reset_index()
        .sort_values('manual_cc_after_dropoff_count', ascending=False)
    )

    return result


# Run the function
loads_with_late_manual_cc = get_loads_with_manual_cc_after_dropoff(df)
qualifying_loads_after = len(loads_with_late_manual_cc)

print(f"Total distinct LOAD_NUMs in dataset:               {total_loads:,}")
print(f"LOAD_NUMs with manual CC after drop off close:     {qualifying_loads_after:,}")
print(f"Percentage:                                        {qualifying_loads_after / total_loads * 100:.1f}%")
print(f"Total Manual CCs after Drop Off Close:             {loads_with_late_manual_cc['manual_cc_after_dropoff_count'].sum():,}")
loads_with_late_manual_cc.head(10)


Total distinct LOAD_NUMs in dataset:               311,086
LOAD_NUMs with manual CC after drop off close:     898
Percentage:                                        0.3%
Total Manual CCs after Drop Off Close:             1,040


,LOAD_NUM,last_dropoff_close,manual_cc_after_dropoff_count,latest_manual_cc
763,554146525,2026-05-23 19:00:00+00:00,9,2026-05-25 12:30:00+00:00
645,553654474,2026-05-23 14:00:00+00:00,9,2026-05-23 19:31:00+00:00
412,552901255,2026-05-13 20:00:00+00:00,8,2026-05-15 11:18:00+00:00
215,552282398,2026-05-05 22:00:00+00:00,8,2026-05-06 12:35:00+00:00
825,554495801,2026-05-27 17:00:00+00:00,7,2026-05-29 11:25:00+00:00
394,552859642,2026-05-09 07:00:00+00:00,7,2026-05-10 14:58:00+00:00
148,552046963,2026-05-08 02:00:00+00:00,6,2026-05-08 13:08:00+00:00
173,552153035,2026-05-06 22:00:00+00:00,6,2026-05-07 13:11:00+00:00
781,554260994,2026-05-25 16:00:00+00:00,6,2026-05-28 09:37:00+00:00
188,552196199,2026-05-05 16:00:00+00:00,5,2026-05-06 13:18:00+00:00


In [9]:
def get_manual_cc_after_last_auto(df):
    """
    Identifies per load how many manual check calls occurred AFTER the last
    automated check call AND BEFORE the last D-Close.

    These represent manual touches that were required because automated tracking
    stopped firing before the load was delivered — an opportunity to reduce
    over-processing by improving automation continuity.

    Criteria for a manual check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == False
        - UPDATE_USER != 'DATASCIENCE'
        - HUMAN_ENTERED_CHECKCALL_FLAG == 1

    Criteria for an automated check call:
        - CHECK_CALL_TYPE == 'CC'
        - AUTOMATED == True
        - HUMAN_ENTERED_CHECKCALL_FLAG == 0
        - UPDATE_USER == 'DATASCIENCE'

    Window of interest per load:
        - Start: last automated CC datetime (exclusive)
        - End:   last D-Close datetime (inclusive)

    Only loads that have both an automated CC and a D-Close are included.
    Loads where the last auto CC is already after the last D-Close are excluded.

    Args:
        df (pd.DataFrame): Raw dataframe from Snowflake.

    Returns:
        pd.DataFrame: One row per qualifying load with:
            - LOAD_NUM
            - last_auto_cc: datetime of the last automated CC on the load
            - last_dropoff_close: datetime of the last D-Close on the load
            - auto_gap_minutes: minutes between last auto CC and last D-Close
            - manual_cc_in_gap_count: manual CCs in the gap (after last auto, before D-Close)
            - earliest_gap_manual_cc: datetime of the first manual CC in the gap
    """
    df = df.copy()
    df['ENTERED_DATETIME_CST'] = pd.to_datetime(df['ENTERED_DATETIME_CST'], utc=True, errors='coerce')

    # --- Last automated CC per load ---
    last_auto = (
        df[
            (df['CHECK_CALL_TYPE'] == 'CC') &
            (df['AUTOMATED'] == True) &
            (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 0) &
            (df['UPDATE_USER'] != 'DATASCIENCE')
        ]
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_auto_cc')
        .reset_index()
    )

    # --- Last D-Close per load ---
    last_dropoff = (
        df[df['CHECK_CALL_TYPE'] == 'D-Close']
        .groupby('LOAD_NUM')['ENTERED_DATETIME_CST']
        .max()
        .rename('last_dropoff_close')
        .reset_index()
    )

    # --- Combine: only loads where last auto CC is BEFORE last D-Close ---
    bounds = last_auto.merge(last_dropoff, on='LOAD_NUM', how='inner')
    bounds = bounds[bounds['last_auto_cc'] < bounds['last_dropoff_close']].copy()
    bounds['auto_gap_minutes'] = (
        (bounds['last_dropoff_close'] - bounds['last_auto_cc'])
        .dt.total_seconds() / 60
    ).round(1)

    # --- Manual CCs ---
    manual_cc = df[
        (df['CHECK_CALL_TYPE'] == 'CC') &
        (df['AUTOMATED'] == False) &
        (df['UPDATE_USER'] != 'DATASCIENCE') &
        (df['HUMAN_ENTERED_CHECKCALL_FLAG'] == 1)
    ][['LOAD_NUM', 'ENTERED_DATETIME_CST']].copy()

    # Keep only manual CCs that fall strictly after the last auto CC and before/at D-Close
    manual_cc = manual_cc.merge(bounds, on='LOAD_NUM', how='inner')
    manual_cc_in_gap = manual_cc[
        (manual_cc['ENTERED_DATETIME_CST'] > manual_cc['last_auto_cc']) &
        (manual_cc['ENTERED_DATETIME_CST'] <= manual_cc['last_dropoff_close'])
    ]

    # --- Aggregate per load ---
    result = (
        manual_cc_in_gap.groupby(['LOAD_NUM', 'last_auto_cc', 'last_dropoff_close', 'auto_gap_minutes'])
        .agg(
            manual_cc_in_gap_count=('ENTERED_DATETIME_CST', 'count'),
            earliest_gap_manual_cc=('ENTERED_DATETIME_CST', 'min')
        )
        .reset_index()
        .sort_values('manual_cc_in_gap_count', ascending=False)
    )

    return result, bounds


# Run the function
manual_cc_after_last_auto, gap_bounds = get_manual_cc_after_last_auto(df)
loads_with_auto_gap = len(manual_cc_after_last_auto)
total_gap_manual = manual_cc_after_last_auto['manual_cc_in_gap_count'].sum()
avg_gap_minutes = gap_bounds['auto_gap_minutes'].median()

print(f"Total distinct LOAD_NUMs in dataset:               {total_loads:,}")
print(f"Loads where last auto CC precedes D-Close:         {len(gap_bounds):,}")
print(f"Loads with manual CCs filling the auto gap:        {loads_with_auto_gap:,}")
print(f"  % of gap loads with manual fill-in:              {loads_with_auto_gap / len(gap_bounds) * 100:.1f}%")
print(f"Total manual CCs in auto gap:                      {total_gap_manual:,}")
print(f"Median gap duration (last auto CC → D-Close):      {avg_gap_minutes:.0f} min")
manual_cc_after_last_auto.head(10)


Total distinct LOAD_NUMs in dataset:               311,086
Loads where last auto CC precedes D-Close:         182,309
Loads with manual CCs filling the auto gap:        993
  % of gap loads with manual fill-in:              0.5%
Total manual CCs in auto gap:                      1,259
Median gap duration (last auto CC → D-Close):      255 min


,LOAD_NUM,last_auto_cc,last_dropoff_close,auto_gap_minutes,manual_cc_in_gap_count,earliest_gap_manual_cc
303,552313222,2026-05-04 20:19:00+00:00,2026-05-07 23:00:00+00:00,4481.0,19,2026-05-04 23:31:00+00:00
941,554871037,2026-05-26 17:26:00+00:00,2026-05-28 23:00:00+00:00,3214.0,13,2026-05-26 23:45:00+00:00
601,553360137,2026-05-14 22:29:00+00:00,2026-05-15 22:00:00+00:00,1411.0,12,2026-05-15 00:21:00+00:00
321,552404766,2026-05-04 22:51:00+00:00,2026-05-06 16:00:00+00:00,2469.0,11,2026-05-05 03:30:00+00:00
262,552223612,2026-05-13 22:41:00+00:00,2026-05-18 15:30:00+00:00,6769.0,8,2026-05-14 04:53:00+00:00
880,554311761,2026-05-24 12:30:00+00:00,2026-05-26 15:00:00+00:00,3030.0,6,2026-05-24 19:47:00+00:00
356,552527278,2026-05-05 17:59:00+00:00,2026-05-06 21:00:00+00:00,1621.0,6,2026-05-05 23:58:00+00:00
23,550784132,2026-05-03 00:09:00+00:00,2026-05-04 17:00:00+00:00,2451.0,5,2026-05-03 02:06:00+00:00
320,552404168,2026-05-04 20:35:00+00:00,2026-05-08 20:00:00+00:00,5725.0,5,2026-05-05 16:15:00+00:00
894,554429980,2026-05-22 15:21:00+00:00,2026-05-23 00:00:00+00:00,519.0,4,2026-05-22 17:33:00+00:00
